[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesamKhallaf73/pytorch-ground-up-notebooks/blob/main/notebooks/L06.ipynb)

# Lecture 06 — Permute, Transpose & Broadcasting

*Two ways axes line up: reorder the ones you have, then stretch across the ones you don’t.*

**PyTorch from Ground Up · Volume I · Chapters 8–9** — companion notebook.

**How to use it:** run the cells from top to bottom. Before each code cell, *predict* what it will print — then run it and compare with the expected output (click ▸ to open it). Every exercise from the lecture is at the end.

By the end you will be able to:
- Swap two axes with `transpose` / `.T` and reorder all of them with `permute`.
- Convert images between HWC and CHW, and make the transformer heads swap.
- State the three broadcasting rules and predict any output shape.
- Recognise when broadcasting fails — and the silent blow-up when it doesn’t.
- Explain why broadcasting is free (a stride of 0).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print('PyTorch', torch.__version__)

## 01 · Reshape moves numbers; transpose moves axes

Reshaping never changes **which axis is which**. Sometimes that’s exactly what you need. Swap height and width; move the channel axis from first to last. `transpose` and `permute` reorder the axes themselves.

<sub>Source: lecture slides 3–8</sub>

**`transpose.py`** — predict the output, then run.

In [ ]:
m = torch.arange(1, 9).reshape(2, 4)
print(m.transpose(0, 1))
print(m.transpose(0, 1).shape)
print(m.transpose(0, 1).is_contiguous())

<details><summary>▸ Expected output</summary>

```
tensor([[1, 5],
        [2, 6],
        [3, 7],
        [4, 8]])
torch.Size([4, 2])
False
```
</details>

> ⚠️ **Watch out — .T is 2-D only**  
> For a 2-D tensor, `.T` is shorthand for `transpose(0, 1)`. On a higher-dimensional tensor, `.T` reverses **all** the axes — almost never what you want. Beyond 2-D, use `transpose` or `permute` explicitly.
>
> Name exactly which axes you mean.

## 02 · Permute is the general tool



<sub>Source: lecture slides 9–12, 19</sub>

**`permute.py`** — predict the output, then run.

In [ ]:
t = torch.randn(2, 3, 4)   # (A, B, C)
p = t.permute(2, 0, 1)     # (C, A, B)
print(p.shape)

# Equivalence: transpose is a special case
a = t.transpose(0, 2)
b = t.permute(2, 1, 0)
print(torch.equal(a, b))   # True

<details><summary>▸ Expected output</summary>

```
torch.Size([4, 2, 3])
True
```
</details>

🧪 *Interactive lab in the app:* **Reorder the axes** — try it there to see this idea move.

## 03 · Images and transformers: where permute earns its keep

Load → permute → run → permute back: **nearly every image pipeline** does this. The permutes are free views.

<sub>Source: lecture slides 13–18</sub>

**`hwc_chw.py`** — predict the output, then run.

In [ ]:
# HWC image from PIL / NumPy
hwc = torch.randn(224, 224, 3)   # (H, W, C)
chw = hwc.permute(2, 0, 1)       # (C, H, W)
print(chw.shape)
# And back again, for plotting
back = chw.permute(1, 2, 0)      # (H, W, C)
print(back.shape)

<details><summary>▸ Expected output</summary>

```
torch.Size([3, 224, 224])
torch.Size([224, 224, 3])
```
</details>

> ⚠️ **Watch out — Make it contiguous before view**  
> After `permute` or `transpose` the tensor is non-contiguous. If a later operation needs contiguous data — `view`, or saving to a file — call `.contiguous()` first.
>
> Common pattern: `img.permute(1, 2, 0).contiguous()`

**`mha_reshape.py`** — predict the output, then run.

In [ ]:
# Typical multi-head reshape
# (batch, seq, heads, d) → (batch, heads, seq, d)
x = torch.randn(2, 10, 4, 64)
x = x.permute(0, 2, 1, 3)   # swap seq and heads
print(x.shape)

<details><summary>▸ Expected output</summary>

```
torch.Size([2, 4, 10, 64])
```
</details>

## 04 · Broadcasting: different shapes, no complaint

A column of 3 plus a row of 4 — and PyTorch doesn’t complain. It **stretches** both. That automatic stretching is **broadcasting**, and it copies no numbers.

<sub>Source: lecture slides 21–27</sub>

**`broadcast.py`** — predict the output, then run.

In [ ]:
a = torch.tensor([[1], [2], [3]])   # (3, 1)
b = torch.tensor([[10,20,30,40]])   # (1, 4)
c = a + b
print(c.shape)
print(c)

<details><summary>▸ Expected output</summary>

```
torch.Size([3, 4])
tensor([[11, 21, 31, 41],
        [12, 22, 32, 42],
        [13, 23, 33, 43]])
```
</details>

**The Broadcasting Rules**

1. **Align from the right.** Line the shapes up from the last dimension. If one tensor has fewer dimensions, pad it with 1s on the left.
2. **Each pair must match.** For each aligned pair, the sizes must be equal, or one of them must be 1.
3. **Size-1 dims stretch.** Any dimension of size 1 is virtually repeated to match the other tensor’s size along that axis.

🧪 *Interactive lab in the app:* **Broadcasting calculator** — try it there to see this idea move.

## 05 · When it fails — and the silent blow-up

If a pair is neither equal nor 1, PyTorch raises — and **that error protects you**. The dangerous case is the one that *doesn’t* error:

<sub>Source: lecture slides 28–31, 34</sub>

**`fails.py`** — predict the output, then run.

In [ ]:
a = torch.ones(3)
b = torch.ones(5)
# a + b → RuntimeError!
# dims: 3 vs 5 — neither is 1

**`blowup.py`** — predict the output, then run.

In [ ]:
a = torch.randn(3)      # (3,)
b = torch.randn(3, 1)   # (3, 1)
c = a - b
print(c.shape)          # (3, 3) ← probably wrong

<details><summary>▸ Expected output</summary>

```
torch.Size([3, 3])
```
</details>

> ⚠️ **Watch out — Element-wise needs exact shapes**  
> A `(3,)` tensor and a `(3, 1)` tensor do **not** subtract element-wise — they broadcast to `(3, 3)`: `(3,)` became `(1, 3)`, `(3, 1)` stayed. No error, just quietly wrong — the number-one broadcasting bug.
>
> Print `.shape` after every operation until the habit is automatic.

**`checklist.py`** — predict the output, then run.

In [ ]:
# 1. Print both shapes
print(a.shape, b.shape)
# 2. Align from the right on paper
#    a: 3, 1
#    b: 1, 4
#  out: 3, 4  ← is this what you want?
# 3. If not, unsqueeze or reshape
#    to control which dim stretches

## 06 · Why it’s free: a stride of 0

**12 virtual, 4 real.** A stride of 0 means “don’t advance — reuse the same row.” That’s why broadcasting stays fast even when the virtual tensor would be enormous.

<sub>Source: lecture slides 32–33</sub>

**`stride_zero.py`** — predict the output, then run.

In [ ]:
a = torch.ones(1, 4)
e = a.expand(3, 4)    # explicit broadcast
print(e.shape)        # (3, 4)
print(e.stride())     # (0, 1) — stride 0!
print(e.numel())      # 12 "virtual" elements
print(a.numel())      # 4 actual elements

<details><summary>▸ Expected output</summary>

```
torch.Size([3, 4])
(0, 1)
12
4
```
</details>

## The rule — worth remembering

**Part One — reordering axes**

1. `transpose(d0, d1)` swaps two axes; `.T` is shorthand for 2-D only.
2. `permute(d0, d1, …)` reorders all axes at once — the general tool.
3. All three return non-contiguous views. No data is copied; strides change.
4. `view` will fail on the result — use `reshape` or `.contiguous().view()`.
5. HWC → CHW: `img.permute(2, 0, 1)`. CHW → HWC: `img.permute(1, 2, 0)`.
6. In transformers, `permute(0, 2, 1, 3)` swaps the seq and heads axes.

**Part Two — broadcasting**

1. Align from the right. Pad shorter shapes with 1s on the left.
2. Each pair: sizes must be equal or one must be 1. Otherwise it errors.
3. Size-1 dims stretch to match — the stretch is virtual, no memory copied.
4. The output shape takes the max of each aligned pair.
5. The biggest danger is a silent blow-up — always print shapes.
6. Internally, a stride of 0 implements the stretch.

## Pause and try

Work each exercise in the empty cell below it. Open the worked answer only after you have tried.

### Exercise 01 · Permute and check

Create a (2, 3, 5) tensor. Permute it to (3, 5, 2). Print the shape and check `.is_contiguous()`. Call `.contiguous()` and check again.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

`t.permute(1, 2, 0)` gives `torch.Size([3, 5, 2])` and `is_contiguous()` is False. After `.contiguous()` it is True — the data was copied into the new order.
</details>

### Exercise 02 · Axes

Load a fake HWC image `torch.randn(100, 80, 3)`, convert it to CHW for PyTorch, then back to HWC for plotting; print all three shapes. Then permute a (4, 8, 2, 64) tensor to (4, 2, 8, 64) — what tuple did you use?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

(100, 80, 3) → `.permute(2, 0, 1)` → (3, 100, 80) → `.permute(1, 2, 0)` → (100, 80, 3). The second tuple is `(0, 2, 1, 3)` — the transformer heads swap.
</details>

### Exercise 03 · Broadcasting

Subtract a (3,) and a (3, 1) tensor — is the result (3,) or (3, 3)? Explain with the alignment rules. Then normalize each row of a (4, 5) matrix to zero mean (hint: mean shape (4, 1)).

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

(3, 3). Aligned from the right, (3,) is padded to (1, 3) and paired with (3, 1): each pair contains a 1, so both stretch. For the rows: `m - m.mean(dim=1, keepdim=True)` — the (4, 1) mean stretches across the 5 columns.
</details>

### Interview-style

Given a NumPy image (H, W, 3), walk through the exact steps to prepare it for a PyTorch CNN (dtype and batch dim included).

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

Convert with `torch.from_numpy(img)`, cast to float32 (and scale from 0–255 to 0–1 if needed) with `.float() / 255`, reorder the axes from HWC to CHW with `.permute(2, 0, 1)`, then add a batch dimension with `.unsqueeze(0)` to get (1, 3, H, W).
</details>

### Interview-style

Explain the broadcasting rules, then give an example of a silent broadcasting bug.

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

Shapes are aligned from the right, padding the shorter one with 1s on the left; each aligned pair must be equal or contain a 1; size-1 dimensions are virtually stretched to match, and the output takes the max of each pair.

A classic silent bug: subtracting a (3,) prediction from a (3, 1) target. No error is raised, but the result is a (3, 3) matrix instead of three element-wise differences — and a loss computed from it is quietly wrong.
</details>